# Feature Engineering + KPI Business 

Objectif : transformer Notre master dataset en dataset analytique exploitable pour insights + dashboard + RFM.

In [1]:
import pandas as pd

In [5]:
master = pd.read_csv("../Data/processed/master_orders.csv")

#### 1. COPIE DE TRAVAIL

In [6]:
df = master.copy()

In [8]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,total_payment,payment_installments,payment_type,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,4.0,500.0,19.0,8.0,13.0,housewares,38.71,1.0,credit_card,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,1.0,400.0,19.0,13.0,19.0,perfumery,141.46,1.0,boleto,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,1.0,420.0,24.0,19.0,21.0,auto,179.12,3.0,credit_card,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,3.0,450.0,30.0,10.0,20.0,pet_shop,72.20,1.0,credit_card,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,4.0,250.0,51.0,15.0,15.0,stationery,28.62,1.0,credit_card,5.0


### 2. FEATURE ENGINEERING TEMPS

Delivery time (temps réel de livraison)

In [9]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col])

In [13]:
df["delivery_time_days"] = (
    df["order_delivered_customer_date"] - df["order_purchase_timestamp"]
).dt.days

Delay (retard livraison)

In [16]:
df["delay_days"] = (
    df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]
).dt.days

Mois / année (analyse business)

In [17]:
df["order_month"] = df["order_purchase_timestamp"].dt.to_period("M")
df["order_year"] = df["order_purchase_timestamp"].dt.year

#### 3. FEATURE REVENUE

Revenue total par ligne

In [21]:
df["total_revenue"] = df["price"] + df["freight_value"]

Revenue net (plus business-oriented)

In [23]:
df["net_revenue"] = df["total_payment"]

#### 4. CUSTOMER FEATURES 

Nombre de commandes par client

In [24]:
customer_orders = df.groupby("customer_unique_id")["order_id"].nunique().reset_index()
customer_orders.columns = ["customer_unique_id", "total_orders"]

df = df.merge(customer_orders, on="customer_unique_id", how="left")

Valeur client totale

In [26]:
customer_value = df.groupby("customer_unique_id")["total_payment"].sum().reset_index()
customer_value.columns = ["customer_unique_id", "customer_lifetime_value"]

df = df.merge(customer_value, on="customer_unique_id", how="left")

In [28]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,payment_type,review_score,delivery_time_days,delay_days,order_month,order_year,total_revenue,net_revenue,total_orders,customer_lifetime_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,credit_card,4.0,8.0,-8.0,2017-10,2017,38.71,38.71,2,82.82
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,boleto,4.0,13.0,-6.0,2018-07,2018,141.46,141.46,1,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,credit_card,5.0,9.0,-18.0,2018-08,2018,179.12,179.12,1,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,credit_card,5.0,13.0,-13.0,2017-11,2017,72.20,72.20,1,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,credit_card,5.0,2.0,-10.0,2018-02,2018,28.62,28.62,1,28.62


#### 5. SATISFACTION FEATURES

Buckets de satisfaction

In [30]:
def satisfaction(score):
    if score >= 5:
        return "Excellent"
    elif score >= 4:
        return "Good"
    elif score >= 3:
        return "Average"
    else:
        return "Poor"

df["satisfaction_level"] = df["review_score"].apply(satisfaction)

#### 6. DELIVERY ANALYSIS FEATURES

Delivery status

In [32]:
df["delivery_status"] = df["delay_days"].apply(
    lambda x: "Late" if x > 0 else "On time or early"
)

#### 7. KPI BUSINESS (VALIDATION RAPIDE)

Total Revenue :

In [33]:
df["total_revenue"].sum()

np.float64(15843553.24)

Total Orders :

In [34]:
df["order_id"].nunique()

99441

Total Customers :

In [35]:
df["customer_unique_id"].nunique()

96096

Average Order Value (AOV) :

In [36]:
df["net_revenue"].sum() / df["order_id"].nunique()

np.float64(205.85801289206668)

#### 8. INSIGHT RAPIDE

Impact du retard sur satisfaction :

In [37]:
df.groupby("delivery_status")["review_score"].mean()

delivery_status
Late                2.257411
On time or early    4.135914
Name: review_score, dtype: float64

Top categories revenue

In [38]:
df.groupby("product_category_name_english")["net_revenue"].sum().sort_values(ascending=False).head(10)

product_category_name_english
bed_bath_table           1712553.67
health_beauty            1657373.12
computers_accessories    1585330.45
furniture_decor          1430176.39
watches_gifts            1429216.68
sports_leisure           1392127.56
housewares               1094758.13
auto                      852294.33
garden_tools              838280.75
cool_stuff                779698.00
Name: net_revenue, dtype: float64

In [39]:
df.to_csv(
    "../data/processed/master_orders_features.csv",
    index=False
)